# 06 - Sample experiment per-phase report

**Audience:** analyzing recordings from the bundled SampleExperiment.

The SampleExperiment cycles four phases (FreeExploration, VisualSearch,
CountingTask, ChangeDetection). `analyze_sample_experiment` slices the CSV
by phase and produces fixation count, scanpath length, gaze entropy, and
LHIPA per phase, joined with per-trial outcomes from the matching
`experiment_results_*.json`.

## What you'll get

- Per-phase metrics in a tidy DataFrame.
- Per-trial outcomes (acquisition times, counting answers, change-detection
  accuracy).
- Phase-segmented scanpath plot.
- Per-phase LHIPA bar chart.

## Getting your data into this notebook

`notebook_context()` auto-discovers a CSV in this order:

1. **Explicit path** — `notebook_context(csv="path/to/EyeTracking_*.csv")`
2. **Environment variable** — `export EYELEAN_CSV=path/to/file.csv`
3. **`Logs/` folder** — most-recent main `EyeTracking_*.csv` in any
   `Logs/` directory walking up from this notebook.
4. **Bundled sample** — falls back to a v1.2 SampleExperiment recording
   shipped in `Assets/StreamingAssets/` so the notebook runs end-to-end
   on a fresh checkout before you've recorded anything yourself.

## 0. Bootstrap

In [ ]:
import os
import sys
from pathlib import Path

# Allow opening this notebook from a checkout without `pip install -e`.
_here = Path(os.getcwd()).resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "eyelean_analysis" / "__init__.py").is_file():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import eyelean_analysis as ela

ctx = ela.notebook_context()
print(ctx)

## 1. Build the report

In [ ]:
report = ela.analyze_sample_experiment(
    ctx.csv_path, results_json_path=ctx.results_path,
)
print(f"Total: {report.total_samples} samples / {report.total_duration_seconds:.1f}s @ {report.sample_rate_hz:.1f} Hz")
print(f"Phases: {list(report.phases.keys())}")
report.to_dataframe()

## 2. Per-trial outcomes (if results JSON was found)

In [ ]:
if ctx.results_path is None:
    print("No experiment_results_*.json next to the CSV; skipping per-trial section.")
else:
    import json
    with open(ctx.results_path) as f:
        outcomes = json.load(f)
    if isinstance(outcomes, dict):
        for k, v in outcomes.items():
            print(f"  {k}: {v if not isinstance(v, list) else f'[{len(v)} entries]'}")
    else:
        print(json.dumps(outcomes, indent=2)[:1500])

## 3. Phase-segmented scanpath

In [ ]:
df = ctx.data.df
if "phase" not in df.columns or "combined_dir_x" not in df.columns:
    print("Missing phase or gaze columns.")
else:
    dx = df["combined_dir_x"].to_numpy(dtype=float)
    dy = df["combined_dir_y"].to_numpy(dtype=float)
    dz = df["combined_dir_z"].to_numpy(dtype=float)
    yaw = np.degrees(np.arctan2(dx, dz))
    pitch = -np.degrees(np.arcsin(np.clip(dy, -1, 1)))
    fig, ax = plt.subplots(figsize=(10, 6))
    for phase_name, sub in df.groupby("phase"):
        idx = sub.index.to_numpy()
        ax.plot(yaw[idx], pitch[idx], linewidth=0.5, alpha=0.7, label=phase_name)
    ax.set_xlabel("yaw (deg)")
    ax.set_ylabel("pitch (deg)")
    ax.invert_yaxis()
    ax.set_aspect("equal", adjustable="datalim")
    ax.set_title("Scanpath by phase")
    ax.legend(loc="upper right", fontsize=8)
    plt.tight_layout()
    plt.show()

## 4. Per-phase metric bar charts

In [ ]:
report_df = report.to_dataframe()
if report_df.empty:
    print("Report is empty (no phases detected).")
else:
    for col in ("n_fixations", "mean_fixation_duration_ms", "lhipa", "gaze_entropy"):
        if col not in report_df.columns:
            continue
        vals = report_df[col]
        if vals.isna().all():
            continue
        fig, ax = plt.subplots(figsize=(8, 2.6))
        ax.bar(report_df["phase"], vals)
        ax.set_ylabel(col)
        ax.set_title(f"{col} by phase")
        plt.xticks(rotation=20, ha="right")
        plt.tight_layout()
        plt.show()